# D1 channel-drop — keep top-k highest-variance channels

Defender keeps only the top-k EEG channels by training-set variance, drops the rest, and retrains the BCI decoder on the smaller channel set. Sweeps k ∈ {64, 32, 16, 8}. k=64 is the no-defense baseline.

Companion to D1 PCA and D1 noise.

Send back `results/11_d1_channel_drop.json` and `runs/<run_id>/meta.json`.

**Runtime → L4 GPU.** Expected wall ~50 min.

**Don't `Save a copy in GitHub` from Colab.**

## 1. Clone repo

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. Prefetch PhysioNet imagery (~2 min)

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

## 5. Run D1 channel-drop sweep — k ∈ {64, 32, 16, 8}

Expected wall: ~50 min.

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.11_d1_adhoc --transform channel_drop --all

## 6. Emit run metadata + result JSON

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
EXPERIMENT = 'experiments.11_d1_adhoc'
ARGS = ['--transform', 'channel_drop', '--all']
RESULTS = 'results/11_d1_channel_drop.json'
FIGURE = 'figures/11_d1_channel_drop.pdf'
TAG = 'd1_channel_drop'
try:
    git_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR).decode().strip()
except Exception as exc:
    print(f'  git unavailable ({exc}); using "unknown"'); git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS,
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS, FIGURE]}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f: json.dump(meta, f, indent=2)
print('=== run metadata ==='); print(json.dumps(meta, indent=2)); print()
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing — Cell 5 did not finish. Re-run Cell 5, then this cell.')
print(f'=== {RESULTS} ==='); print(open(RESULTS).read())

## 7. Download artifacts

In [ ]:
from google.colab import files
files.download('results/11_d1_channel_drop.json')
files.download(f'runs/{run_id}/meta.json')